In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
import gc
import subprocess
import os
import time
from openai import OpenAI
import time
import pandas as pd
from tqdm import tqdm
import nltk
nltk.download('wordnet')
nltk.download('punkt')
import time
import pandas as pd
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import BERTScorer
import sacrebleu

[nltk_data] Downloading package wordnet to /home/timur/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/timur/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Инициализация BERTScore (один раз, чтобы не пересоздавать модель)
bertscorer = BERTScorer(lang="ru", rescale_with_baseline=False)
def compute_metrics(reference, hypothesis):
    """
    Вычисляет все метрики для одной пары (эталон, ответ модели).
    reference : str – эталонный ответ
    hypothesis : str – ответ модели
    возвращает словарь с метриками
    """
    # Normalize: убираем лишние пробелы, приводим к нижнему регистру? 
    # Для Exact Match лучше не менять регистр, если важно точное совпадение.
    # Для остальных метрик обычно нормализуют.
    ref = reference.strip()
    hyp = hypothesis.strip()
    
    # Exact Match (полное совпадение строк)
    exact_match = int(ref == hyp)
    
    # BLEU (используем smoothing, т.к. предложения короткие)
    smoothing = SmoothingFunction().method1
    # BLEU ожидает список токенов, токенизируем по пробелам (простая токенизация)
    ref_tokens = ref.split()
    hyp_tokens = hyp.split()
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing)
    
    # ROUGE (возьмём ROUGE-L для баланса)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    scores = scorer.score(ref, hyp)
    rouge_l = scores['rougeL'].fmeasure
    
    # METEOR (требует токенизации, используем простую)
    meteor = meteor_score([ref.split()], hyp.split())
    
    # BERTScore (возвращает Precision, Recall, F1; берём F1)
    P, R, F1 = bertscorer.score([hyp], [ref])
    bertscore_f1 = F1.item()
    
    return {
        "exact_match": exact_match,
        "bleu": bleu,
        "rouge_l": rouge_l,
        "meteor": meteor,
        "bertscore_f1": bertscore_f1
    }


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def api_ai_answer(client, prompt):
    start_time = time.perf_counter()
    response = client.chat.completions.create(
        model="LORA_THE_BEST",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1600,
        temperature=0.0
    )
    
    end_time = time.perf_counter()
    
    # Извлекаем текст ответа
    answer = response.choices[0].message.content
    response_time = end_time - start_time
    completion_tokens = response.usage.completion_tokens
    tokens_per_second = completion_tokens / response_time if response_time > 0 else 0
    return [answer, response_time, completion_tokens,tokens_per_second]

In [4]:
def test_model_adapter(base_model_name, adapter_path,
              command = [
    "vllm", "serve",
    "LORA_THE_BEST",
    "--host", "0.0.0.0",
    "--port", "8001",
    "--api-key", "7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51",
    "--served-model-name", "LORA_THE_BEST",
    "--max-model-len", "8192",
    "--max-num-seqs", "4"
]):

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token

    model = PeftModel.from_pretrained(base_model, adapter_path)
    merged_model = model.merge_and_unload()
    output_dir = "LORA_THE_BEST"
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    del merged_model
    del model
    del base_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    #print(torch.cuda.memory_summary())

    log_file = open(f"DUMPS_VLLM/vllm_server_{adapter_path}.log", "w")

    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,   # объединяем stderr с stdout
        text=True,                  # работаем со строками
        start_new_session=True      # отделяем процесс, чтобы он не завершился при выходе скрипта
    )
    #time.sleep(40)
    
    # Ждём строку "Application startup complete" в логе
    ready = False
    while not ready:
        time.sleep(1)
        with open(f"DUMPS_VLLM/vllm_server_{adapter_path}.log", "r") as f:
            if "Application startup complete" in f.read():
                ready = True
                break
    client = OpenAI(
        base_url="http://localhost:8001/v1",
        api_key="7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51"
    )

    #start_time = time.perf_counter()
    #response = client.chat.completions.create(
    #    model="LORA_THE_BEST",
    #    messages=[{"role": "user", "content": "Мой кариотип 46,XX,t(5;10)(q12;p22). Что это значит?"}],
    #    max_tokens=1600,
    #    temperature=0.7
    #)
    #end_time = time.perf_counter()
    #answer = response.choices[0].message.content
    #print("Ответ модели:")
    #print(answer)
    #print(f"\nВремя выполнения: {end_time - start_time:.3f} секунд")
    
    df = pd.read_excel("Q_F_T.xlsx")

    outputs = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Обработка запросов"):
        prompt = row['Promts']
        reference = row['References']

        answer, resp_time, comp_tokens, tps = api_ai_answer(client, prompt)
        metrics = compute_metrics(reference, answer)
        
        # Сохраняем всё вместе
        outputs.append({
            "answer": answer,
            "response_time": resp_time,
            "completion_tokens": comp_tokens,
            "tokens_per_second": tps,
            **metrics
        })

    # Распаковываем в DataFrame
    answers = [o["answer"] for o in outputs]
    response_times = [o["response_time"] for o in outputs]
    completion_tokens = [o["completion_tokens"] for o in outputs]
    tokens_per_second = [o["tokens_per_second"] for o in outputs]
    exact_match = [o["exact_match"] for o in outputs]
    bleu = [o["bleu"] for o in outputs]
    rouge_l = [o["rouge_l"] for o in outputs]
    meteor = [o["meteor"] for o in outputs]
    bertscore_f1 = [o["bertscore_f1"] for o in outputs]
    
    # Добавляем в исходный df
    df["Answers"] = answers
    df["Response times"] = response_times
    df["Completion tokens"] = completion_tokens
    df["Tokens per second"] = tokens_per_second
    df["Exact Match"] = exact_match
    df["BLEU"] = bleu
    df["ROUGE-L"] = rouge_l
    df["METEOR"] = meteor
    df["BERTScore F1"] = bertscore_f1
    
    # Сохраняем результат
    df.to_excel(f"TESTS_VLLM/VLLM_TEST_with_metrics_adapter_{adapter_path}.xlsx", index=False)
    
    process.terminate()
    

In [5]:
apdpter_path_dict = {
    "meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_3_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head" : "meta-llama/Llama-3.1-8B-Instruct",
    
}

for k, v in apdpter_path_dict.items():
    print(k, v)
    test_model_adapter(v, k)

`torch_dtype` is deprecated! Use `dtype` instead!


meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_3_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Обработка запросов: 100%|██████████| 105/105 [11:17<00:00,  6.45s/it]
